# 07 Options RV

**Book:** *Fixed Income Relative Value Analysis (2nd ed.)*  
**Focus:** Chapter 19 scaffold for single-underlying option relative value and vega-sector PCA.


## Goals

1. Define a notebook structure for option RV despite missing local option-surface data.
2. Reuse local underlying and rate proxies where possible.
3. Separate single-underlying option RV from vega-sector factor analysis.


In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

REPO_ROOT = Path("/Users/zelin/Desktop/PA Investment/Invest_strategy")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from quant_data.api import get_data

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

START = "2024-02-26"
END = "2026-02-27"
BOOK_PDF = Path(r"""/Users/zelin/Desktop/阅读学习/Fixed Income Relative Value Analysis + Website A Practitioner’s Guide to the Theory, Tools, and Trades 2nd.pdf""")

print("Expected environment: conda activate ibkr-analytics && export PYTHONPATH=.")
print("Book PDF exists:", BOOK_PDF.exists())
print("Repo root:", REPO_ROOT)


In [ ]:
FRED_DIR = REPO_ROOT / "data" / "market_data" / "fred"
PRICES_DIR = REPO_ROOT / "data" / "market_data" / "prices"


def _filter_date(frame: pd.DataFrame, start=START, end=END, date_col="date"):
    out = frame.copy()
    out[date_col] = pd.to_datetime(out[date_col])
    return out[(out[date_col] >= start) & (out[date_col] <= end)]


def load_fred_series(series_ids, start=START, end=END):
    frames = []
    for parquet_file in sorted(FRED_DIR.glob("*.parquet")):
        df = pd.read_parquet(parquet_file)
        if "series_id" not in df.columns:
            continue
        sub = df[df["series_id"].isin(series_ids)]
        if not sub.empty:
            frames.append(_filter_date(sub, start=start, end=end))

    if not frames:
        return pd.DataFrame()

    joined = pd.concat(frames, ignore_index=True).drop_duplicates(["date", "series_id"])
    wide = (
        joined.pivot(index="date", columns="series_id", values="value")
        .sort_index()
        .apply(pd.to_numeric, errors="coerce")
    )
    wide.index = pd.to_datetime(wide.index)
    return wide


def load_local_price_series(tickers, start=START, end=END, value_col="close"):
    frames = []
    for parquet_file in sorted(PRICES_DIR.glob("*.parquet")):
        df = pd.read_parquet(parquet_file)
        if "ticker" not in df.columns or value_col not in df.columns:
            continue
        sub = df[df["ticker"].isin(tickers)]
        if not sub.empty:
            frames.append(_filter_date(sub, start=start, end=end))

    if not frames:
        return pd.DataFrame()

    joined = pd.concat(frames, ignore_index=True).drop_duplicates(["date", "ticker"])
    wide = (
        joined.pivot(index="date", columns="ticker", values=value_col)
        .sort_index()
        .apply(pd.to_numeric, errors="coerce")
    )
    wide.index = pd.to_datetime(wide.index)
    return wide


In [ ]:
underlyings = load_local_price_series(["^IRX", "^FVX", "^TNX", "^TYX"])
macro = load_fred_series(["SOFR", "DFEDTARU", "DGS2", "DGS10"]).dropna()

underlyings.tail(), macro.tail()


In [ ]:
underlyings.plot(title="Available Local Underlying Proxies for Rates/Options RV")
plt.show()


## Single-underlying option RV scaffold

Chapter 19 appears to separate option RV into several types. For a single-underlying workflow, the typical structure is:

- choose one underlying and one part of the surface
- compare observed option prices or implied vols with a model or relative-value benchmark
- isolate theta / vega / skew exposures
- test whether the residual mispricing mean reverts


In [ ]:
single_underlying_schema = pd.DataFrame(
    {
        "field": [
            "date",
            "underlying",
            "expiry",
            "strike",
            "option_type",
            "mid_price",
            "implied_vol",
            "delta",
            "vega",
        ],
        "status": [
            "required",
            "required",
            "required",
            "required",
            "required",
            "required",
            "required",
            "useful",
            "useful",
        ],
    }
)
single_underlying_schema


In [ ]:
# TODO: load actual option-surface data when available.
# TODO: define a fair-value benchmark (surface fit, relative vol pair, or model-implied vol).
# TODO: compute residual mispricing and test its persistence / mean reversion.


## Vega sector PCA scaffold

The chapter's later option workflow appears to apply PCA to a vega sector or volatility surface slice.

Use this section to:

- arrange the implied-vol surface into a tenor/strike or tenor/delta matrix
- compute PCA on changes in implied vol
- identify dominant vega factors
- construct factor-neutral option RV structures


In [ ]:
vega_sector_schema = pd.DataFrame(
    {
        "dimension": ["observation_date", "expiry_bucket", "delta_bucket", "implied_vol", "vega_weight"],
        "notes": [
            "time index",
            "term structure bucket",
            "moneyness or delta bucket",
            "surface level to analyze",
            "weighting field for PCA or exposure control",
        ],
    }
)
vega_sector_schema


In [ ]:
# TODO: once option surface data exists, reshape into a matrix such as:
# rows   -> dates
# cols   -> expiry/delta buckets
# values -> implied vol changes or vega-weighted vol changes
# Then run PCA and inspect factor stability / residual dislocations.


## Data gaps

Local fixed-income option data is not currently in the lake. Missing inputs include:

- implied volatility surfaces
- option mid prices
- Greeks or sufficient inputs to compute them
- realized-vol benchmarks aligned to the option universe

This notebook is intentionally a structure scaffold until those datasets are ingested.
